<a href="https://colab.research.google.com/github/kvssri/online-tihiitg/blob/main/final%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Final Model / Pipeline Freeze

import pandas as pd
import numpy as np
import os
import zipfile
import glob

print("Final Model / Pipeline Freeze")
print("Notebook started successfully.")

Final Model / Pipeline Freeze
Notebook started successfully.


In [2]:
from google.colab import files

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

print("Uploaded file:", zip_file)

Saving Underwater-Image-Data-set-main.zip to Underwater-Image-Data-set-main.zip
Uploaded file: Underwater-Image-Data-set-main.zip


In [3]:
extract_path = "/content/final_model_data"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

csv_files = glob.glob(
    extract_path + "/**/*.csv",
    recursive=True
)

print("CSV files found:", len(csv_files))

for f in csv_files:
    print(os.path.basename(f))

CSV files found: 8
training_log 2.csv
training_log 4.csv
training_log 6.csv
training_log 8.csv
training_log 1.csv
training_log 5.csv
training_log 7.csv
training_log 3.csv


In [4]:
all_logs = []

for f in csv_files:
    temp = pd.read_csv(f)
    temp["source_file"] = os.path.basename(f)
    all_logs.append(temp)

df = pd.concat(all_logs, ignore_index=True)

print("Final dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nRows per source:")
print(df["source_file"].value_counts())

Final dataset shape: (24915, 6)

Columns:
['epoch', 'step', 'gen_total', 'disc_loss', 'time_s', 'source_file']

Rows per source:
source_file
training_log 4.csv    3900
training_log 7.csv    3900
training_log 5.csv    3900
training_log 1.csv    3900
training_log 3.csv    3900
training_log 2.csv    3660
training_log 6.csv    1521
training_log 8.csv     234
Name: count, dtype: int64


In [5]:
# ================================
# FROZEN FINAL PIPELINE SETTINGS
# ================================

FEATURES = ["epoch", "step", "disc_loss", "time_s"]
TARGET = "gen_total"

SEQUENCE_LENGTH = 10

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

HIDDEN_CHANNELS = 64
DROPOUT = 0.0
LEARNING_RATE = 0.0005
BATCH_SIZE = 32

EPOCHS = 15
RANDOM_SEED = 42

print("Final pipeline settings frozen.")
print("--------------------------------")
print("Features:", FEATURES)
print("Target:", TARGET)
print("Sequence length:", SEQUENCE_LENGTH)
print("Train / Validation / Test:", "70% / 15% / 15%")
print("Hidden channels:", HIDDEN_CHANNELS)
print("Dropout:", DROPOUT)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Training epochs:", EPOCHS)

Final pipeline settings frozen.
--------------------------------
Features: ['epoch', 'step', 'disc_loss', 'time_s']
Target: gen_total
Sequence length: 10
Train / Validation / Test: 70% / 15% / 15%
Hidden channels: 64
Dropout: 0.0
Learning rate: 0.0005
Batch size: 32
Training epochs: 15


In [6]:
# Sort data consistently
df = df.sort_values(
    ["source_file", "epoch", "step"]
).reset_index(drop=True)

# Create sequences
X_all = df[FEATURES].values
y_all = df[TARGET].values

X_seq = []
y_seq = []

for i in range(len(X_all) - SEQUENCE_LENGTH):
    X_seq.append(
        X_all[i:i + SEQUENCE_LENGTH]
    )
    y_seq.append(
        y_all[i + SEQUENCE_LENGTH]
    )

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("Sequence construction completed.")
print("--------------------------------")
print("Sequence input shape:", X_seq.shape)
print("Target shape:", y_seq.shape)

Sequence construction completed.
--------------------------------
Sequence input shape: (24905, 10, 4)
Target shape: (24905,)


In [7]:
# ==========================================
# FROZEN TIME-ORDERED DATA SPLIT
# ==========================================

n = len(X_seq)

train_end = int(TRAIN_RATIO * n)
val_end = int((TRAIN_RATIO + VALIDATION_RATIO) * n)

X_train = X_seq[:train_end]
y_train = y_seq[:train_end]

X_val = X_seq[train_end:val_end]
y_val = y_seq[train_end:val_end]

X_test = X_seq[val_end:]
y_test = y_seq[val_end:]

print("Frozen data split")
print("---------------------------")
print("Training samples   :", len(X_train))
print("Validation samples :", len(X_val))
print("Test samples       :", len(X_test))

print("\nInput shapes:")
print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

Frozen data split
---------------------------
Training samples   : 17433
Validation samples : 3736
Test samples       : 3736

Input shapes:
Train: (17433, 10, 4)
Val  : (3736, 10, 4)
Test : (3736, 10, 4)


In [8]:
from sklearn.preprocessing import StandardScaler

# ==========================================
# FROZEN TRAINING-ONLY NORMALIZATION
# ==========================================

scaler = StandardScaler()

# Fit ONLY on training data
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
scaler.fit(X_train_2d)

# Transform all splits using the training-fitted scaler
X_train = scaler.transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_val = scaler.transform(
    X_val.reshape(-1, X_val.shape[-1])
).reshape(X_val.shape)

X_test = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

print("Final normalization frozen.")
print("Scaler fitted only on training data.")

print("\nTraining normalized mean:")
print(X_train.mean(axis=(0, 1)))

print("\nTraining normalized standard deviation:")
print(X_train.std(axis=(0, 1)))

Final normalization frozen.
Scaler fitted only on training data.

Training normalized mean:
[-1.78447797e-15 -5.00503098e-16 -6.90367284e-16 -4.56635381e-16]

Training normalized standard deviation:
[1. 1. 1. 1.]


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ==========================================
# FROZEN FINAL TCN ARCHITECTURE
# ==========================================

class TCNModel(nn.Module):
    def __init__(self, input_size, hidden_channels=64, dropout=0.0):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(
                input_size,
                hidden_channels,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Conv1d(
                hidden_channels,
                hidden_channels,
                kernel_size=3,
                padding=2,
                dilation=2
            ),
            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Conv1d(
                hidden_channels,
                32,
                kernel_size=3,
                padding=4,
                dilation=4
            ),
            nn.ReLU()
        )

        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.network(x)
        x = x[:, :, -1]
        return self.fc(x)


# Create final frozen model
final_model = TCNModel(
    input_size=X_train.shape[2],
    hidden_channels=HIDDEN_CHANNELS,
    dropout=DROPOUT
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LEARNING_RATE
)

# Convert data to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Final TCN created successfully.")
print("Parameters:", sum(p.numel() for p in final_model.parameters()))

Final TCN created successfully.
Parameters: 19393


In [10]:
id="q7n3x1"
import copy
import time
import random

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_model = final_model.to(device)

best_val_loss = float("inf")
best_state = None

train_losses = []
val_losses = []

start_time = time.time()

for epoch in range(EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    final_model.train()
    total_train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = final_model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * X_batch.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    # -------------------------
    # Validation
    # -------------------------
    final_model.eval()
    total_val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = final_model(X_batch)
            loss = criterion(predictions, y_batch)

            total_val_loss += loss.item() * X_batch.size(0)

    val_loss = total_val_loss / len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Save best validation model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(final_model.state_dict())

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

# Restore best validation checkpoint
final_model.load_state_dict(best_state)

runtime = time.time() - start_time

print("\nFinal training completed.")
print("Best validation loss:", best_val_loss)
print("Training time:", round(runtime, 2), "seconds")
print("Device:", device)

Epoch 01/15 | Train Loss: 71.099450 | Val Loss: 20.870800
Epoch 02/15 | Train Loss: 41.952944 | Val Loss: 19.927651
Epoch 03/15 | Train Loss: 38.827841 | Val Loss: 19.922915
Epoch 04/15 | Train Loss: 37.459960 | Val Loss: 19.919537
Epoch 05/15 | Train Loss: 36.502126 | Val Loss: 19.768732
Epoch 06/15 | Train Loss: 35.725602 | Val Loss: 19.646511
Epoch 07/15 | Train Loss: 34.898336 | Val Loss: 19.606654
Epoch 08/15 | Train Loss: 34.266245 | Val Loss: 19.699224
Epoch 09/15 | Train Loss: 33.806234 | Val Loss: 19.732659
Epoch 10/15 | Train Loss: 33.443115 | Val Loss: 19.643887
Epoch 11/15 | Train Loss: 33.456436 | Val Loss: 19.490660
Epoch 12/15 | Train Loss: 32.981447 | Val Loss: 19.678773
Epoch 13/15 | Train Loss: 32.449128 | Val Loss: 19.801264
Epoch 14/15 | Train Loss: 32.511524 | Val Loss: 19.728661
Epoch 15/15 | Train Loss: 32.354913 | Val Loss: 19.509676

Final training completed.
Best validation loss: 19.490660261035732
Training time: 59.3 seconds
Device: cpu


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Put model in evaluation mode
final_model.eval()

X_test_device = X_test_tensor.to(device)

with torch.no_grad():
    predictions = final_model(X_test_device).cpu().numpy().flatten()

actual = y_test_tensor.numpy().flatten()

# Calculate metrics
mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))
mape = np.mean(
    np.abs((actual - predictions) / np.where(actual == 0, 1e-8, actual))
) * 100
r2 = r2_score(actual, predictions)

print("FINAL TEST RESULTS")
print("-------------------")
print("MAE  :", round(mae, 6))
print("RMSE :", round(rmse, 6))
print("MAPE :", round(mape, 4), "%")
print("R²   :", round(r2, 6))

# Save predictions
final_predictions = pd.DataFrame({
    "Actual_gen_total": actual,
    "Predicted_gen_total": predictions,
    "Absolute_Error": np.abs(actual - predictions)
})

final_predictions.to_csv(
    "/content/final_predictions.csv",
    index=False
)

# Save metrics
final_metrics = pd.DataFrame([{
    "Model": "Final Tuned TCN",
    "Target": TARGET,
    "MAE": mae,
    "RMSE": rmse,
    "MAPE (%)": mape,
    "R2": r2,
    "Best_Validation_Loss": best_val_loss,
    "Training_Time_seconds": runtime
}])

final_metrics.to_csv(
    "/content/final_metrics.csv",
    index=False
)

print("\nSaved:")
print("/content/final_predictions.csv")
print("/content/final_metrics.csv")

FINAL TEST RESULTS
-------------------
MAE  : 5.938097
RMSE : 7.525253
MAPE : 36.1976 %
R²   : -1.233077

Saved:
/content/final_predictions.csv
/content/final_metrics.csv


In [12]:
import torch

# Save final trained model
model_path = "/content/final_tcn_model.pth"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "input_size": X_train.shape[2],
        "hidden_channels": HIDDEN_CHANNELS,
        "dropout": DROPOUT,
        "sequence_length": SEQUENCE_LENGTH,
        "target": TARGET,
        "features": FEATURES
    },
    model_path
)

print("Final model saved successfully.")
print("File:", model_path)

Final model saved successfully.
File: /content/final_tcn_model.pth


In [13]:
import joblib

# Save the training-only scaler
scaler_path = "/content/final_scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Final scaler saved successfully.")
print("File:", scaler_path)

Final scaler saved successfully.
File: /content/final_scaler.pkl


In [14]:
import json

# Final frozen configuration
final_config = {
    "project": "Underwater Image Enhancement - GAN Log Time-Series Experiment",
    "target": TARGET,
    "features": FEATURES,
    "sequence_length": SEQUENCE_LENGTH,

    "data_split": {
        "train_ratio": TRAIN_RATIO,
        "validation_ratio": VALIDATION_RATIO,
        "test_ratio": TEST_RATIO,
        "split_method": "time-ordered"
    },

    "preprocessing": {
        "scaler": "StandardScaler",
        "scaler_fit": "training_data_only"
    },

    "model": {
        "architecture": "TCN",
        "hidden_channels": HIDDEN_CHANNELS,
        "dropout": DROPOUT
    },

    "training": {
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "loss_function": "MSELoss",
        "optimizer": "Adam",
        "random_seed": RANDOM_SEED
    },

    "selection_basis": {
        "selected_model": "Tuned TCN",
        "selected_configuration": "TCN-4",
        "day19_best_validation_loss": 22.040364
    },

    "evaluation_metrics": [
        "MAE",
        "RMSE",
        "MAPE",
        "R2"
    ]
}

config_path = "/content/final_pipeline_config.json"

with open(config_path, "w") as f:
    json.dump(final_config, f, indent=4)

print("Final pipeline configuration saved.")
print("File:", config_path)

Final pipeline configuration saved.
File: /content/final_pipeline_config.json


In [15]:
# Save training and validation loss history

training_history = pd.DataFrame({
    "Epoch": range(1, EPOCHS + 1),
    "Training_Loss": train_losses,
    "Validation_Loss": val_losses
})

history_path = "/content/final_training_history.csv"

training_history.to_csv(
    history_path,
    index=False
)

print("Training history saved successfully.")
print("File:", history_path)

print("\nBest validation loss:")
print(round(min(val_losses), 6))
print("Best epoch:")
print(np.argmin(val_losses) + 1)

Training history saved successfully.
File: /content/final_training_history.csv

Best validation loss:
19.49066
Best epoch:
11


In [16]:
rationale = f"""
FINAL MODEL / PIPELINE FREEZE — TECHNICAL SELECTION RATIONALE

Project:
Underwater Image Enhancement — GAN Log Time-Series Experiment

1. FINAL MODEL SELECTION
The final model selected is the Tuned TCN (TCN-4).

TCN-4 configuration:
- Hidden channels: {HIDDEN_CHANNELS}
- Dropout: {DROPOUT}
- Learning rate: {LEARNING_RATE}
- Batch size: {BATCH_SIZE}
- Sequence length: {SEQUENCE_LENGTH}
- Epochs: {EPOCHS}

TCN-4 was selected during the controlled hyperparameter experiments
because it achieved the best validation loss among the tested TCN
configurations.

Best validation loss from the controlled experiment:
22.040364

2. PREPROCESSING
- Features: {FEATURES}
- Target: {TARGET}
- StandardScaler normalization
- Scaler fitted only on training data
- Sequence length: {SEQUENCE_LENGTH}

3. DATA SPLIT
The frozen split is:
- Training: 70%
- Validation: 15%
- Testing: 15%

The split is time-ordered so that future observations are not used
for training.

4. EVALUATION
The final model is evaluated on the held-out test set using:
- MAE
- RMSE
- MAPE
- R2

5. ROBUSTNESS CONSIDERATION
The project-specific robustness analysis compared performance across
available source/training runs. No verified cell IDs or explicit
degradation-stage labels were available in the provided data.

Therefore, source_file was treated as a source/training-run identifier,
rather than claiming it represents a physical battery cell or official
degradation stage.

6. FINAL TEST RESULT
The final test results are stored in:
final_metrics.csv

The individual predictions and errors are stored in:
final_predictions.csv

7. FROZEN ARTEFACTS
The following artefacts are saved:
- final_tcn_model.pth
- final_scaler.pkl
- final_pipeline_config.json
- final_training_history.csv
- final_predictions.csv
- final_metrics.csv

8. IMPORTANT DATA SCOPE
The available dataset contains GAN training logs with the target
gen_total. It does not contain verified SoH or RUL labels.
Therefore, this final experiment predicts gen_total and should not
be presented as a battery SoH/RUL model.

FINAL DECISION:
The Tuned TCN is frozen as the final candidate because it provided the
strongest validation evidence among the tested TCN configurations,
while maintaining a relatively compact architecture and fixed,
reproducible preprocessing and evaluation methodology.
"""

rationale_path = "/content/final_model_selection_rationale.txt"

with open(rationale_path, "w") as f:
    f.write(rationale)

print("Final selection rationale saved successfully.")
print("File:", rationale_path)

Final selection rationale saved successfully.
File: /content/final_model_selection_rationale.txt


In [17]:
import os

final_files = [
    "/content/final_tcn_model.pth",
    "/content/final_scaler.pkl",
    "/content/final_pipeline_config.json",
    "/content/final_training_history.csv",
    "/content/final_predictions.csv",
    "/content/final_metrics.csv",
    "/content/final_model_selection_rationale.txt"
]

print("FINAL ARTEFACT CHECK")
print("====================")

all_found = True

for file_path in final_files:
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"✓ {os.path.basename(file_path)} — {size_kb:.2f} KB")
    else:
        print(f"✗ MISSING: {os.path.basename(file_path)}")
        all_found = False

print("\n====================")

if all_found:
    print("SUCCESS: All final artefacts are present.")
else:
    print("WARNING: One or more artefacts are missing.")

FINAL ARTEFACT CHECK
✓ final_tcn_model.pth — 79.75 KB
✓ final_scaler.pkl — 0.69 KB
✓ final_pipeline_config.json — 1.03 KB
✓ final_training_history.csv — 0.61 KB
✓ final_predictions.csv — 105.58 KB
✓ final_metrics.csv — 0.20 KB
✓ final_model_selection_rationale.txt — 2.23 KB

SUCCESS: All final artefacts are present.


In [18]:
import zipfile
import os

final_artifacts = [
    "/content/final_tcn_model.pth",
    "/content/final_scaler.pkl",
    "/content/final_pipeline_config.json",
    "/content/final_training_history.csv",
    "/content/final_predictions.csv",
    "/content/final_metrics.csv",
    "/content/final_model_selection_rationale.txt"
]

zip_path = "/content/Final_Model_Pipeline_Freeze_Artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in final_artifacts:
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=os.path.basename(file_path))

print("Final artefact ZIP created successfully.")
print("File:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024, 2), "KB")

Final artefact ZIP created successfully.
File: /content/Final_Model_Pipeline_Freeze_Artifacts.zip
Size: 124.91 KB
